# Etapa 01 — Denoising de ECG (sinais reais MIT-BIH)

Notebook didático da Etapa 01. Trabalhamos com **sinais reais** da MIT-BIH
(sem ruído sintético) e comparamos um filtro FIR de fase linear como técnica
principal contra três alternativas: IIR Butterworth, média móvel e filtragem
direta no domínio da frequência. Também mostramos o equivalente FIR via FFT
(convolução rápida).

Para tornar a comparação concreta, escolhemos **três registos** com
morfologias clinicamente distintas e, em cada um, isolamos a **janela mais
ruidosa** que contém ao menos uma anotação do símbolo de interesse:

| Registro | Símbolo | Significado |
|---------|---------|-------------|
| 100 | N | Batimento normal sinusal (referência limpa) |
| 109 | L | Bloqueio de ramo esquerdo (LBBB) |
| 118 | R | Bloqueio de ramo direito (RBBB) |

A duração da janela é controlada pela variável global `SAMPLE_DURATION_SEC`
na célula 1.2. **Mude esse valor para experimentar com janelas maiores ou
menores** — o resto do notebook se adapta automaticamente.

> Justificativas teóricas, frequências de corte, ordens dos filtros e
> métricas estão detalhadas em
> [`docs/etapa_01_denoising.md`](../docs/etapa_01_denoising.md).


## Sumário

1. Configuração e parâmetros globais
2. Seleção de segmentos reais com símbolo-alvo e ruído máximo
3. Visão geral dos três segmentos brutos
4. Projeto dos filtros FIR de fase linear
5. FIR isolado por tipo de ruído (no segmento mais ruidoso)
6. Cadeia FIR completa (HP → BS → LP) nos 3 segmentos
7. FIR via FFT — convolução rápida e equivalência
8. Alternativa: IIR Butterworth
9. Alternativa: média móvel
10. Alternativa: filtragem direta no domínio da frequência
11. Tabela consolidada de comparação


## 1. Configuração e setup


### 1.1. Imports


In [ ]:
from __future__ import annotations

import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wfdb

from src.config import mitdb_record_dir, PROCESSED_DATA_DIR
from src.preprocessing import fir_filters as fir
from src.preprocessing import iir_filters as iir
from src.preprocessing import simple_filters as simp
from src.preprocessing import metrics as M
from src.preprocessing.segments import extract_segment, Segment
from src.visualization.ecg_plots import plot_overlay, plot_psd, plot_filter_response

warnings.filterwarnings("ignore", category=RuntimeWarning)
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (11, 3.4)})

DATA_DIR = mitdb_record_dir()
print("Diretório de dados:", DATA_DIR)


### 1.2. Parâmetros globais

Estes são os parâmetros que controlam todo o resto do notebook. Mude
`SAMPLE_DURATION_SEC` para tornar a amostra de estudo maior ou menor (em
segundos). O canal é o índice WFDB do canal a analisar (0 corresponde
geralmente a MLII).


In [ ]:
SAMPLE_DURATION_SEC: float = 10.0

# canal 0
CHANNEL: int = 0

# Registos e símbolos-alvo. Cada (registo, símbolo) gera UMA janela do tamanho acima.
RECORDS_OF_INTEREST: list[tuple[str, str, str]] = [
    ("100", "N", "Normal sinusal"),
    ("109", "L", "Bloqueio de ramo esquerdo (LBBB)"),
    ("118", "R", "Bloqueio de ramo direito (RBBB)"),
]

# Frequências e parâmetros dos filtros (ver docs/etapa_01_denoising.md §3).
HP_CUT_HZ: float = 0.5      # baseline wander
HP_TRANS_HZ: float = 1.0
PLI_LO_HZ: float = 59.0     # rejeita-faixa para 60 Hz
PLI_HI_HZ: float = 61.0
PLI_TRANS_HZ: float = 1.0
LP_CUT_HZ: float = 40.0     # ruído muscular
LP_TRANS_HZ: float = 8.0

# Banda passante (referência para métrica de preservação)
PASSBAND: tuple[float, float] = (1.0, 30.0)

# Bandas-alvo (para métricas de redução)
BAND_BW: tuple[float, float] = (0.0, 0.5)
BAND_PLI: tuple[float, float] = (58.0, 62.0)
BAND_EMG: tuple[float, float] = (40.0, 90.0)


## 2. Seleção de segmentos reais

Para cada par (registo, símbolo-alvo), extraímos a janela de
`SAMPLE_DURATION_SEC` segundos que (i) contém pelo menos uma anotação do
símbolo e (ii) maximiza um indicador heurístico de ruído (razão entre
potência fora-da-banda 0,05–0,8 Hz + 40–90 Hz + 58–62 Hz e potência útil em
1–30 Hz). Assim olhamos sempre para o pior trecho dentro do registo,
enfatizando o efeito da filtragem.


In [ ]:
def load_record(record_id: str) -> tuple[np.ndarray, float, np.ndarray, np.ndarray]:
    rec = wfdb.rdrecord((DATA_DIR / record_id).as_posix())
    ann = wfdb.rdann((DATA_DIR / record_id).as_posix(), "atr")
    return rec.p_signal[:, CHANNEL], float(rec.fs), ann.sample, np.asarray(ann.symbol)


segments: list[Segment] = []
for rid, sym, _label in RECORDS_OF_INTEREST:
    x_full, fs, ann_samples, ann_syms = load_record(rid)
    seg = extract_segment(
        rid, x_full, fs, ann_samples, ann_syms,
        target_symbol=sym, duration_sec=SAMPLE_DURATION_SEC, channel=CHANNEL,
    )
    segments.append(seg)

FS = segments[0].fs
assert all(abs(s.fs - FS) < 1e-9 for s in segments), "fs deve ser uniforme entre registos"
print(f"fs = {FS:g} Hz, janela = {SAMPLE_DURATION_SEC:g} s ({len(segments[0].x)} amostras)")
for seg, (_rid, _sym, label) in zip(segments, RECORDS_OF_INTEREST):
    print(
        f"rec {seg.record_id} ({label}): "
        f"start={seg.start_sample}, fim={seg.end_sample}, "
        f"alvos={len(seg.target_indices)}, score_ruído={seg.noise_score:.3f}"
    )


## 3. Visão geral dos segmentos brutos

Cada subplot abaixo mostra a janela bruta do registo correspondente.
Os triângulos vermelhos marcam as anotações WFDB de batimento dentro da
janela. Note como o registo 118 (R) apresenta deriva e ruído visível, ao
passo que o 100 (N) é praticamente limpo — base para entender por que a
filtragem é mais ou menos crítica em cada caso.


In [ ]:
fig, axes = plt.subplots(len(segments), 1, figsize=(11, 2.8 * len(segments)), sharex=True)
for ax, seg, (_rid, sym, label) in zip(axes, segments, RECORDS_OF_INTEREST):
    ax.plot(seg.t, seg.x, color="#1f77b4", lw=0.8)
    ax.scatter(seg.annotations_time, np.full(len(seg.annotations_samples), seg.x.max() * 1.05),
               marker="v", color="#d62728", s=18, label="anotações")
    ax.set_title(f"Registo {seg.record_id} — {label}  |  score ruído = {seg.noise_score:.3f}")
    ax.set_ylabel("mV")
    ax.grid(alpha=0.3)
axes[-1].set_xlabel("tempo (s)")
plt.tight_layout()
plt.show()


## 4. Projeto dos filtros FIR de fase linear

Aplicamos o método da janela com janela de Hamming. O número de coeficientes
segue `N ≈ 3,3 · fs / Δf` (arredondado para ímpar, garantindo Tipo I). Os
parâmetros aqui valem para `fs = 360 Hz`:

| Filtro | Tipo | Corte | Δf | N (taps) |
|--------|------|-------|----|----------|
| BW | passa-alta | 0,5 Hz | 1,0 Hz | ≈ 1189 |
| PLI | rejeita-faixa | 59–61 Hz | 1,0 Hz | ≈ 1189 |
| EMG | passa-baixa | 40 Hz | 8,0 Hz | ≈ 149 |

A justificativa detalhada das frequências de corte e ordens está em
[`docs/etapa_01_denoising.md`](../docs/etapa_01_denoising.md).


In [ ]:
h_hp = fir.design_highpass(FS, HP_CUT_HZ, transition_hz=HP_TRANS_HZ)
h_bs = fir.design_bandstop(FS, PLI_LO_HZ, PLI_HI_HZ, transition_hz=PLI_TRANS_HZ)
h_lp = fir.design_lowpass(FS, LP_CUT_HZ, transition_hz=LP_TRANS_HZ)
print(f"FIR HP : N = {len(h_hp)}  (atraso {(len(h_hp)-1)//2 / FS*1000:.1f} ms antes de filtfilt)")
print(f"FIR BS : N = {len(h_bs)}  (atraso {(len(h_bs)-1)//2 / FS*1000:.1f} ms antes de filtfilt)")
print(f"FIR LP : N = {len(h_lp)}  (atraso {(len(h_lp)-1)//2 / FS*1000:.1f} ms antes de filtfilt)")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
plot_filter_response(axes[0], FS, h=h_hp, title=f"FIR HP @ {HP_CUT_HZ} Hz", xlim=(0, 5))
plot_filter_response(axes[1], FS, h=h_bs, title=f"FIR BS {PLI_LO_HZ}–{PLI_HI_HZ} Hz", xlim=(50, 70))
plot_filter_response(axes[2], FS, h=h_lp, title=f"FIR LP @ {LP_CUT_HZ} Hz", xlim=(0, 80))
for ax in axes:
    ax.set_ylim(-120, 5)
plt.tight_layout()
plt.show()


## 5. FIR isolado por tipo de ruído

Para mostrar o efeito de cada filtro por si só, aplicamos cada um
isoladamente ao **segmento mais ruidoso** (registo 118). Cada subplot mostra
o sinal cru em cinza ao fundo e o filtrado em cor à frente; ao lado,
a PSD (Welch) antes e depois.


In [ ]:
# pegar o segmento mais ruidoso
seg_focus = max(segments, key=lambda s: s.noise_score)
print(f"Segmento de foco: registo {seg_focus.record_id} (score {seg_focus.noise_score:.3f})")


### 5.1. Passa-alta (remove baseline wander)


In [ ]:
y_hp = fir.apply_filtfilt(h_hp, seg_focus.x)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.2))
plot_overlay(axes[0], seg_focus.t, seg_focus.x, y_hp,
             title=f"Rec {seg_focus.record_id} — FIR HP isolado",
             filt_label=f"HP @ {HP_CUT_HZ} Hz")
plot_psd(axes[1], {"cru": seg_focus.x, "filtrado": y_hp}, FS,
         title="PSD — antes vs depois (HP)", xlim=(0, 5))
plt.tight_layout()
plt.show()

print(M.evaluate_denoising("FIR-HP", seg_focus.x, y_hp, FS, target_band=BAND_BW,
                            passband=PASSBAND, annotations=seg_focus.annotations_samples - seg_focus.start_sample))


### 5.2. Rejeita-faixa (remove interferência da rede 60 Hz)


In [ ]:
y_bs = fir.apply_filtfilt(h_bs, seg_focus.x)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.2))
plot_overlay(axes[0], seg_focus.t, seg_focus.x, y_bs,
             title=f"Rec {seg_focus.record_id} - FIR BS isolado",
             filt_label=f"BS {PLI_LO_HZ}–{PLI_HI_HZ} Hz")
plot_psd(axes[1], {"cru": seg_focus.x, "filtrado": y_bs}, FS,
         title="PSD - antes vs depois (BS)", xlim=(50, 70))
plt.tight_layout()
plt.show()

print(M.evaluate_denoising("FIR-BS", seg_focus.x, y_bs, FS, target_band=BAND_PLI,
                            passband=PASSBAND, annotations=seg_focus.annotations_samples - seg_focus.start_sample))


### 5.3. Passa-baixa (remove EMG)


In [ ]:
y_lp = fir.apply_filtfilt(h_lp, seg_focus.x)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.2))
plot_overlay(axes[0], seg_focus.t, seg_focus.x, y_lp,
             title=f"Rec {seg_focus.record_id} — FIR LP isolado",
             filt_label=f"LP @ {LP_CUT_HZ} Hz")
plot_psd(axes[1], {"cru": seg_focus.x, "filtrado": y_lp}, FS,
         title="PSD — antes vs depois (LP)", xlim=(0, 80))
plt.tight_layout()
plt.show()

print(M.evaluate_denoising("FIR-LP", seg_focus.x, y_lp, FS, target_band=BAND_EMG,
                            passband=PASSBAND, annotations=seg_focus.annotations_samples - seg_focus.start_sample))


## 6. Cadeia FIR completa (HP → BS → LP)

Ordem aplicada conforme PDF do projeto: primeiro removemos BW (componente
de maior amplitude), depois o notch de 60 Hz e por fim o LP em 40 Hz.
Como todos são LTI, comutam matematicamente — a ordem só altera transientes
de borda. Aplicamos a cadeia aos três segmentos.


In [ ]:
def fir_chain(x: np.ndarray) -> np.ndarray:
    y = fir.apply_filtfilt(h_hp, x)
    y = fir.apply_filtfilt(h_bs, y)
    y = fir.apply_filtfilt(h_lp, y)
    return y

results_fir: dict[str, np.ndarray] = {}
fig, axes = plt.subplots(len(segments), 2, figsize=(13, 2.8 * len(segments)))
for row, seg in enumerate(segments):
    y = fir_chain(seg.x)
    results_fir[seg.record_id] = y
    plot_overlay(axes[row, 0], seg.t, seg.x, y,
                 title=f"Rec {seg.record_id} ({seg.target_symbol}) — FIR chain",
                 filt_label="FIR HP→BS→LP")
    plot_psd(axes[row, 1], {"cru": seg.x, "filtrado": y}, FS,
             title="PSD — cru vs FIR chain", xlim=(0, 90))
plt.tight_layout()
plt.show()


## 7. FIR via FFT (convolução rápida)

Para sinais longos, a aplicação direta de `lfilter` é `O(N · M)`, enquanto
`fftconvolve` é `O((N+M) log(N+M))`. Demonstramos a equivalência (mesma
resposta de magnitude e fase linear) usando o filtro LP, que tem N moderado.


In [ ]:
# Equivalência numérica entre fftconvolve e filtfilt? fftconvolve faz
# convolução simples (atraso linear, sem fase zero). Aqui
# aplicamos o FIR LP via fftconvolve mode='same' e comparamos com
# a versão lfilter+compensação. Resultados devem ser numericamente iguais.
y_lfilt = fir.apply_lfilter_compensated(h_lp, seg_focus.x)
y_fft   = fir.apply_fft(h_lp, seg_focus.x)
diff = np.max(np.abs(y_lfilt - y_fft))
print(f"max |lfilter+comp - fftconvolve| = {diff:.3e}  (≈ 0: convolução rápida ≡ FIR no tempo)")

import time
t0 = time.perf_counter(); fir.apply_lfilter_compensated(h_lp, seg_focus.x); t_l = time.perf_counter()-t0
t0 = time.perf_counter(); fir.apply_fft(h_lp, seg_focus.x);             t_f = time.perf_counter()-t0
print(f"tempo lfilter+comp : {t_l*1e3:.2f} ms")
print(f"tempo fftconvolve  : {t_f*1e3:.2f} ms")


## 8. Alternativa: IIR Butterworth (`sosfiltfilt`)

Butterworth ordem 4 em cada estágio, aplicado em fase zero via
`sosfiltfilt`. Vantagem: número de coeficientes drasticamente menor que o
FIR equivalente. Limitação: fase não-linear (mitigada pela aplicação
forward-backward).


In [ ]:
sos_hp = iir.design_highpass_sos(FS, HP_CUT_HZ, order=4)
sos_bs = iir.design_bandstop_sos(FS, PLI_LO_HZ, PLI_HI_HZ, order=4)
sos_lp = iir.design_lowpass_sos(FS, LP_CUT_HZ, order=4)

def iir_chain(x: np.ndarray) -> np.ndarray:
    y = iir.apply_sosfiltfilt(sos_hp, x)
    y = iir.apply_sosfiltfilt(sos_bs, y)
    y = iir.apply_sosfiltfilt(sos_lp, y)
    return y

results_iir: dict[str, np.ndarray] = {}
fig, axes = plt.subplots(len(segments), 2, figsize=(13, 2.8 * len(segments)))
for row, seg in enumerate(segments):
    y = iir_chain(seg.x)
    results_iir[seg.record_id] = y
    plot_overlay(axes[row, 0], seg.t, seg.x, y,
                 title=f"Rec {seg.record_id} — IIR Butterworth chain",
                 filt_label="IIR HP→BS→LP", filt_color="#2ca02c")
    plot_psd(axes[row, 1], {"cru": seg.x, "IIR": y, "FIR": results_fir[seg.record_id]},
             FS, title="PSD — cru / IIR / FIR", xlim=(0, 90))
plt.tight_layout()
plt.show()


## 9. Alternativa: média móvel

Duas operações de MA centradas:

- BW: subtraímos uma MA longa (~1 s) do sinal — comportamento aproximadamente
  passa-alta, mas com lobe lateral significativo;
- PLI: MA de N=6 amostras zera exatamente 60, 120 e 180 Hz em fs=360 Hz —
  é a forma mais barata de notch para essa amostragem específica.

Não há equivalente eficiente para EMG via MA simples; o LP FIR é mantido na
cadeia para fechar a banda. **Por que mostrar isto?** É a baseline mais
ingênua possível e expõe limitações de seletividade que justificam o FIR.


In [ ]:
MA_BW_LEN = max(1, int(round(FS * 1.0)))   # ~1 segundo (estimador de baseline)
MA_PLI_LEN = int(round(FS / 60.0))         # 6 amostras em fs=360 Hz

def ma_chain(x: np.ndarray) -> np.ndarray:
    y = simp.remove_baseline_ma(x, MA_BW_LEN)
    y = simp.pli_notch_via_ma(y, MA_PLI_LEN)
    y = fir.apply_filtfilt(h_lp, y)  # mantém LP FIR para limitar EMG
    return y

results_ma: dict[str, np.ndarray] = {}
fig, axes = plt.subplots(len(segments), 2, figsize=(13, 2.8 * len(segments)))
for row, seg in enumerate(segments):
    y = ma_chain(seg.x)
    results_ma[seg.record_id] = y
    plot_overlay(axes[row, 0], seg.t, seg.x, y,
                 title=f"Rec {seg.record_id} — média móvel",
                 filt_label="MA BW + MA notch + LP FIR", filt_color="#9467bd")
    plot_psd(axes[row, 1], {"cru": seg.x, "MA": y, "FIR": results_fir[seg.record_id]},
             FS, title="PSD — cru / MA / FIR", xlim=(0, 90))
plt.tight_layout()
plt.show()


## 10. Alternativa: filtragem direta no domínio da frequência

FFT do sinal → zerar bins fora da banda passante e na vizinhança de 60 Hz →
IFFT. É *brick-wall*: máxima seletividade espectral, mas com **oscilações
de Gibbs** e transientes nas bordas — limitações que serão evidentes no
overlay.


In [ ]:
def freqdomain_chain(x: np.ndarray) -> np.ndarray:
    return simp.freq_domain_filter(
        x, FS,
        zero_below_hz=HP_CUT_HZ,
        zero_above_hz=LP_CUT_HZ,
        notch_band_hz=(PLI_LO_HZ, PLI_HI_HZ),
    )

results_fd: dict[str, np.ndarray] = {}
fig, axes = plt.subplots(len(segments), 2, figsize=(13, 2.8 * len(segments)))
for row, seg in enumerate(segments):
    y = freqdomain_chain(seg.x)
    results_fd[seg.record_id] = y
    plot_overlay(axes[row, 0], seg.t, seg.x, y,
                 title=f"Rec {seg.record_id} — FFT brick-wall",
                 filt_label="FFT brick-wall", filt_color="#ff7f0e")
    plot_psd(axes[row, 1], {"cru": seg.x, "FFT": y, "FIR": results_fir[seg.record_id]},
             FS, title="PSD — cru / FFT / FIR", xlim=(0, 90))
plt.tight_layout()
plt.show()


## 11. Tabela consolidada

Métricas escolhidas (sem ground truth, pois trabalhamos com sinal real):

- **redução_BW_dB / redução_PLI_dB / redução_EMG_dB** — atenuação em dB na
  banda alvo (positivo = atenuou); medido entre o sinal cru e o filtrado.
- **preservação_passband_dB** — variação na banda 1–30 Hz; ~0 dB indica que
  a morfologia do ECG não foi alterada;
- **picos_R_RMS_ms** — deslocamento RMS dos picos R em relação às anotações
  WFDB (ground truth temporal).

Cada linha é uma combinação (registo, método).


In [ ]:
def metrics_for(name: str, raw: np.ndarray, filt: np.ndarray, ann_local: np.ndarray) -> dict:
    row = {
        "método": name,
        "redução_BW_dB": M.power_reduction_db(raw, filt, FS, *BAND_BW),
        "redução_PLI_dB": M.power_reduction_db(raw, filt, FS, *BAND_PLI),
        "redução_EMG_dB": M.power_reduction_db(raw, filt, FS, *BAND_EMG),
        "preservação_passband_dB": M.passband_change_db(raw, filt, FS, *PASSBAND),
    }
    rp = M.r_peak_shift_ms(ann_local, filt, FS)
    row["picos_R_RMS_ms"] = rp["rms_ms"]
    row["picos_R_max_ms"] = rp["max_ms"]
    row["picos_R_n"] = rp["n"]
    return row


rows = []
methods: dict[str, dict[str, np.ndarray]] = {
    "FIR (HP+BS+LP, filtfilt)": results_fir,
    "IIR Butterworth (sosfiltfilt)": results_iir,
    "Média móvel + LP FIR": results_ma,
    "FFT brick-wall": results_fd,
}
for seg in segments:
    ann_local = seg.annotations_samples - seg.start_sample
    for mname, mres in methods.items():
        row = metrics_for(mname, seg.x, mres[seg.record_id], ann_local)
        row = {"registo": seg.record_id, "símbolo": seg.target_symbol, **row}
        rows.append(row)

df_results = pd.DataFrame(rows)
df_results = df_results.round(3)
display(df_results)

OUT_CSV = PROCESSED_DATA_DIR / "etapa01_denoising_metrics.csv"
df_results.to_csv(OUT_CSV, index=False)
print(f"Tabela exportada em: {OUT_CSV}")


---

### Como interpretar a tabela

- **FIR HP+BS+LP** é a referência: redução forte em todas as bandas alvo,
  preservação próxima de 0 dB e picos R muito estáveis (< 5 ms).
- **IIR** alcança números semelhantes com ordens muito menores; pode
  introduzir transientes maiores nas bordas em sinais curtos.
- **Média móvel** atenua menos PLI quando o período não é múltiplo exato
  (mas em fs=360 Hz com N=6 funciona bem); a filtragem BW por subtração de
  MA é menos seletiva e pode atenuar conteúdo lento útil.
- **FFT brick-wall** dá a maior atenuação aparente em PSD, mas o overlay
  tipicamente mostra ondulações de Gibbs próximas a transições, o que
  desloca picos R e degrada `picos_R_RMS_ms`.

A discussão didática completa está em
[`docs/etapa_01_denoising.md`](../docs/etapa_01_denoising.md).
